In [ ]:
import pandas as pd
import numpy as np
import warnings
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
from safeaipackage import check_accuracy, check_robustness, check_explainability, check_fairness, check_privacy  
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import time

warnings.filterwarnings('ignore')
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 10000)

In [ ]:
data = pd.read_excel("D:/files/research_activities/ORGANIZED_FILES/safeaipackage_2/examples/employee.xlsx")
print("This dataset has {} rows and {} columns".format(data.shape[0], data.shape[1]))
data.head()

In [ ]:
data = pd.concat([data,data])
len(data)

In [ ]:
data["gender"] = np.where(data["gender"]=="m", 0, 1)
data["minority"] = np.where(data["minority"]=="no_min", 0, 1)
data = pd.get_dummies(data, columns=["jobcat"])
data.head()

In [ ]:
data["doubling_salary"] = np.where(data["salary"]/data["startsal"] > 2,1,0)

data["doubling_salary"].value_counts()

In [ ]:
X = data.drop(["doubling_salary", "salary", "startsal"], axis=1)
y = data["doubling_salary"]

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=1)

print(xtrain.shape)
print(xtest.shape)

In [ ]:
def _delta_function(data, func):
        result = (func(data.iloc[:,0], data.iloc[:,2]))-(func(data.iloc[:,0], data.iloc[:,1]))
        return result

def _rge_delta_function(data):
    return _den(data.iloc[:,0])-_num(data.iloc[:,0], data.iloc[:,1])

def _rga(y, yhat):
    y = pd.DataFrame(y).reset_index(drop=True)
    yhat = pd.DataFrame(yhat).reset_index(drop=True)
    df = pd.concat([y,yhat], axis=1)
    df.columns = ["y", "yhat"]
    ryhat = yhat.rank(method="min")
    df["ryhat"] = ryhat
    support = df.groupby('ryhat')['y'].mean().reset_index(name='support')
    support_dict = dict(zip(support['ryhat'], support['support']))

    df['rord'] = df['ryhat'].map(support_dict)

    vals = [[i, values] for i, values in enumerate(df["yhat"])]
    ranks = [x[0] for x in sorted(vals, key= lambda item: item[1])]
    ystar = [df['rord'][i] for i in ranks]

    I = list(range(len(y)))
    conc = 2*sum([I[i]*ystar[i] for i in range(len(I))])
    dec= 2*sum([sorted(df["y"], reverse=True)[i]*I[i] for i in range(len(I))]) 
    inc = 2*sum([sorted(df["y"])[i]*I[i] for i in range(len(I))]) 
    RGA=(conc-dec)/(inc-dec)
    return RGA

In [ ]:
class Accuracy:
    def __init__(self, xtrain, xtest, ytrain, ytest, model,n=2, size=500):
        self.samples = []
        self.sample_idx = []
        self.xtrain = pd.DataFrame(xtrain).reset_index(drop=True)
        self.xtest = pd.DataFrame(xtest).reset_index(drop=True)
        self.ytrain = pd.DataFrame(ytrain).reset_index(drop=True)
        self.ytest = pd.DataFrame(ytest).reset_index(drop=True)
        self.model = model
        model_full = self.model.fit(self.xtrain, self.ytrain)
        self.yhat_cm = pd.DataFrame(model_full.predict(self.xtest))
        self.n = n
        self.size = size
        preds_df = pd.concat([self.ytest, self.yhat_cm], axis=1)
        if self.ytest.shape[0]>1000:
            for i in range(n):
                sample = (preds_df.groupby(by=preds_df.columns[0]).sample(size, replace=True))
                self.sample_idx.append(sample.index.values)
                self.samples.append(sample.reset_index(drop=True))    
    def rga(self):
        if self.samples ==[]:
            df = pd.concat([self.ytest, self.yhat_cm], axis=1)
            df.columns = ["y", "yhat"]
            df["ryhat"] = df["yhat"].rank(method="min")

            support = df.groupby('ryhat')['y'].mean().reset_index(name='support')
            support_dict = dict(zip(support['ryhat'], support['support']))

            df['rord'] = df['ryhat'].map(support_dict)

            vals = [[i, values] for i, values in enumerate(df["yhat"])]
            ranks = [x[0] for x in sorted(vals, key= lambda item: item[1])]
            ystar = [df['rord'][i] for i in ranks]

            I = range(len(self.ytest))
            conc = 2 * sum(i * ystar[i] for i in I)
            y_sorted = sorted(df["y"], reverse=True)
            dec = 2 * sum(y_sorted[i] * i for i in I)
            inc = 2 * sum(sorted(df["y"])[i] * i for i in I)
            return (conc - dec) / (inc - dec)
        else:
            RGAs = []
            for sample in self.samples:
                df = sample.reset_index(drop=True)
                df.columns = ["y", "yhat"]
                ryhat = self.yhat_cm.rank(method="min")
                df["ryhat"] = ryhat
                support = df.groupby('ryhat')['y'].mean().reset_index(name='support')
                rord = list(range(len(sample)))
                for jj in range(len(sample)):
                    for ii in range(len(support)):
                            if df["ryhat"][jj]== support['ryhat'][ii]:
                                rord[jj] = support['support'][ii]
                vals = [[i, values] for i, values in enumerate(df["yhat"])]
                ranks = [x[0] for x in sorted(vals, key= lambda item: item[1])]
                ystar = [rord[i] for i in ranks]
                I = list(range(len(sample)))
                conc = 2*sum([I[i]*ystar[i] for i in range(len(I))])
                dec= 2*sum([sorted(df["y"], reverse=True)[i]*I[i] for i in range(len(I))]) 
                inc = 2*sum([sorted(df["y"])[i]*I[i] for i in range(len(I))]) 
                RGAs.append((conc-dec)/(inc-dec))                     
            return "The average of the RGA values for the {} samples is equal to {}".format(self.n, np.mean(RGAs))
    
    
    def rga_statistic_test(self, problemtype, variable= np.nan):
        if problemtype not in ["classification", "prediction"]:
            raise ValueError("problemtype should be classification or prediction")
                             
        if variable is not np.nan: 
            xtrain_rm = self.xtrain.drop(variable, axis=1)
            xtest_rm = self.xtest.drop(variable, axis=1)
            model_rm = self.model.fit(xtrain_rm, self.ytrain)
            yhat_rm = pd.DataFrame(model_rm.predict(xtest_rm))

        else:
            if problemtype == "classification":
                model_2 = DummyClassifier(strategy="most_frequent", random_state=1)
            else:
                model_2 = DummyRegressor(strategy="mean")
                                                
            model_rm = model_2.fit(self.xtrain, self.ytrain)
            yhat_rm = pd.DataFrame(model_rm.predict(self.xtest))

        if self.samples == []:
            #yhat_rm.reset_index(drop= True, inplace=True)
            jk_mat = pd.concat([self.ytest, yhat_rm, self.yhat_cm], axis=1).reset_index(drop=True)
            jk_mat.columns = ["y", "yhat_rm", "yhat_cm"]
            n = len(jk_mat)
            index = np.arange(n)
            jk_results = []
            for i in range(n):
                jk_sample = jk_mat.drop(labels= i)
                jk_sample.reset_index(drop=True, inplace=True)
                jk_statistic = _delta_function(jk_sample, _rga)
                jk_results.append(jk_statistic)
            se = np.sqrt(((n-1)/n)*(sum([(x-np.mean(jk_results))**2 for x in jk_results])))
            z = (_rga(self.ytest,self.yhat_cm)-_rga(self.ytest,yhat_rm))/se
            p_value = 2*scipy.stats.norm.cdf(-abs(z)) 
            return p_value
        else:
            pvalues = []
            for n in range(self.n):
                index = self.sample_idx[n]
                yhat_rm_ = yhat_rm.iloc[index,:]
                yhat_rm_.reset_index(drop=True, inplace=True)
                jk_mat = self.samples[n]
                jk_mat.columns = ["y", "yhat_cm"]
                jk_mat.insert(1, "yhat_rm", yhat_rm_)
                n = len(jk_mat)
                index = np.arange(n)
                jk_results = []
                for i in range(n):
                    jk_sample = jk_mat.drop(labels= i)
                    jk_sample.reset_index(drop=True, inplace=True)
                    jk_statistic = _delta_function(jk_sample, _rga)
                    jk_results.append(jk_statistic)
                se = np.sqrt(((n-1)/n)*(sum([(x-np.mean(jk_results))**2 for x in jk_results])))
                z = (_rga(jk_mat["y"], jk_mat["yhat_cm"])-_rga(jk_mat["y"],jk_mat["yhat_rm"]))/se
                pval = 2*scipy.stats.norm.cdf(-abs(z))
                pvalues.append(pval)
            return np.mean(pvalues)

In [ ]:
rf_model = RandomForestClassifier(random_state=1)
start = time.time()
accuracy_obj = Accuracy(xtrain, xtest, ytrain, ytest, rf_model)
rga = accuracy_obj.rga()
end = time.time()
print(rga)
print(accuracy_obj.rga_statistic_test("classification"))
print(end-start)

In [ ]:
start = time.time()
pk_accuracy = check_accuracy.Accuracy(xtrain, xtest, ytrain, ytest, rf_model)
rga = pk_accuracy.rga()
end = time.time()
print(rga)
print(pk_accuracy.rga_statistic_test("classification"))
print(end-start)